In [ ]:
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd
import random
from matplotlib import pyplot as plt

from utils.geo import haversine_distance

#seed = 14
np.random.seed(random.randrange(1, 100))

n_nodes = 10
k=3
n_simulations = 1
alpha_value = 0.9
alphazero = 9

cell_dataset = pd.read_parquet("assets/porto_cells.parquet")
output = f"TON_IoT_{n_nodes}n_{k}k"
bbox_img = "assets/background.png"
bbox_boundaries = "assets/BBox_Porto.txt"

In [ ]:
with open(bbox_boundaries, "r") as f:
    box = eval(f.readline())
    
img = plt.imread(bbox_img)

def plot_network(network: nx.Graph, path: Path, towers: np.ndarray) -> None:
    fig, ax = plt.subplots(figsize=(6,6))    
    
    ax.set_xlim([box[0], box[1]])
    ax.set_ylim([box[2], box[3]])
    
    aspect_ratio = abs((box[1]-box[0])/(box[2]-box[3]))

    ax.set_aspect(aspect_ratio)

    ax.imshow(img, extent=box, aspect=aspect_ratio)
        
    nx.draw_networkx(
            network,
            pos=towers,
            with_labels=True,
            node_color='gold',
            node_size=150,
            width=1.5,
            ax=ax,
            font_size=8,
            edgecolors="black",
        )
    ax.set_axis_on()

    plt.savefig(path, dpi=500)

def build_network(
    cell_dataset: pd.DataFrame, num_towers: int, k_edge_connectivity: int
) -> tuple[nx.Graph, pd.DataFrame]:
    """
    Builds a network of towers.

    :param cell_dataset: the dataset containing tower positions
    :param num_towers: the number of towers (nodes) to use for the network
    :param k_edge_connectivity: k edge connectivity of the desired graph
    :return: a tuple containing as first element the networkx graph object, as second element a
    dataframe with node coordinates
    """
    while True:  # when a result is found, it is directly returned
        # sample the required number of towers
        towers = cell_dataset.sample(num_towers, replace=False)
        towers.reset_index(drop=True, inplace=True)

        # calculate tuples of node distances (v, w, dist(v,w))
        distances = set()
        for i in range(num_towers):
            for j in range(i + 1, num_towers):
                dist = haversine_distance(
                    longitude_1=towers["lon"].astype(float)[i],
                    longitude_2=towers["lon"].astype(float)[j],
                    latitude_1=towers["lat"].astype(float)[i],
                    latitude_2=towers["lat"].astype(float)[j],
                )
                distances.add((i, j, dist))

        network = nx.Graph()
        network.add_nodes_from(range(num_towers))

        edges = nx.k_edge_augmentation(network, k_edge_connectivity, avail=distances)
        network.add_edges_from(edges)

        return network, towers[["lon", "lat"]]

In [ ]:
for i in range(n_simulations):
    folder_path = Path(f"data/networks/{output}/{i}")
    folder_path.mkdir(parents=True, exist_ok=True)
    folder_path2 = Path(f"data/networks/{output}/{i+1}")
    folder_path2.mkdir(parents=True, exist_ok=True)

    network, towers = build_network(cell_dataset, num_towers=n_nodes, k_edge_connectivity=k)

    plot_network(network, path=folder_path / "plot.jpg", towers=towers.values)
    nx.write_adjlist(network, folder_path / "adj_list.txt")
    towers.to_csv(folder_path / "nodes.csv", index=False)

    plot_network(network, path=folder_path2 / "plot.jpg", towers=towers.values)
    nx.write_adjlist(network, folder_path2 / "adj_list.txt")
    towers.to_csv(folder_path2 / "nodes.csv", index=False)

    print("Saved!")

In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler, LabelEncoder


# Caricamento dataset
def f(i):
    return pd.read_csv(i, 
                       usecols=["ts","src_bytes","missed_bytes","label","type","conn_state", "duration"], 
                       na_values="-"
    )

df = pd.concat(map(f, [ 
                       'assets/Processed_Network_dataset/Network_dataset_2.csv',
                       'assets/Processed_Network_dataset/Network_dataset_3.csv', 
                       'assets/Processed_Network_dataset/Network_dataset_4.csv',
                       'assets/Processed_Network_dataset/Network_dataset_5.csv',
                       'assets/Processed_Network_dataset/Network_dataset_6.csv',
                       'assets/Processed_Network_dataset/Network_dataset_7.csv',
                       'assets/Processed_Network_dataset/Network_dataset_8.csv',
                       'assets/Processed_Network_dataset/Network_dataset_9.csv',
                       'assets/Processed_Network_dataset/Network_dataset_10.csv',
                       'assets/Processed_Network_dataset/Network_dataset_11.csv',
                       'assets/Processed_Network_dataset/Network_dataset_12.csv',
                       'assets/Processed_Network_dataset/Network_dataset_13.csv', 
                       'assets/Processed_Network_dataset/Network_dataset_14.csv', 
                       'assets/Processed_Network_dataset/Network_dataset_15.csv',
                       'assets/Processed_Network_dataset/Network_dataset_16.csv', 
                       'assets/Processed_Network_dataset/Network_dataset_17.csv',
                       'assets/Processed_Network_dataset/Network_dataset_18.csv',
                       'assets/Processed_Network_dataset/Network_dataset_19.csv',
                       'assets/Processed_Network_dataset/Network_dataset_20.csv',
                       'assets/Processed_Network_dataset/Network_dataset_21.csv']))
df = df.dropna()
print("Count: ", df['type'].value_counts())


In [ ]:
target_col = df.columns[-1]
y = df[target_col]
df = df.drop(columns=[target_col])

#separazione colonne
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
boolean_cols = [col for col in df.columns if df[col].dropna().nunique() == 2 and df[col].dropna().isin([0, 1]).all()]
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.difference(boolean_cols).tolist()

df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

scaler = StandardScaler()
cols_to_scale = list(set(numeric_cols) & set(df_encoded.columns))
df_encoded[cols_to_scale] = scaler.fit_transform(df_encoded[cols_to_scale])
x = df_encoded.astype('float32').values
print(x.shape) 

le = LabelEncoder()
y_encoded = le.fit_transform(y)  # ['normal', 'dos', ...] → [0, 1, ...]

mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print(mapping)

In [ ]:
#Dataset creation
import numpy as np
from utils.data import train_val_test_split

def dirichlet_split(y, n_clients, alpha):
    num_c = len(np.unique(y))
    net_dataidx_map = {}
    p_client = np.zeros((n_clients, num_c))
    for i in range(n_clients):
        p_client[i] = np.random.dirichlet(np.repeat(alpha, num_c))
    idx_batch = [[] for _ in range(n_clients)]
    for k in range(num_c):
        idx_k = np.where(y == k)[0]
        np.random.shuffle(idx_k)
        proportions = p_client[:, k]
        proportions = proportions / proportions.sum()
        split_points = (np.cumsum(proportions) * len(idx_k)).astype(int)[:-1]
        idx_batch = [
            idx_j + idx.tolist()
            for idx_j, idx in zip(idx_batch, np.split(idx_k, split_points))
        ]
    for j in range(n_clients):
        np.random.shuffle(idx_batch[j])
        net_dataidx_map[j] = idx_batch[j]
    net_cls_counts = {}
    for net_i, dataidx in net_dataidx_map.items():
        unq, unq_cnt = np.unique(y.iloc[dataidx], return_counts=True)
        net_cls_counts[net_i] = dict(zip(unq, unq_cnt))
    return net_dataidx_map, net_cls_counts


#Y_train_prova = next(iter(train_loader))[1].numpy()
#inds, net_cls_counts = dirichlet_split(Y_train_prova, 10, 0.05, 10)

#print("Inds", inds)
#print("Net cls count", net_cls_counts)

#n_nodes = 10
#k=3
#n_simulations = 1
network_path = Path(f"data/networks/TON_IoT_{n_nodes}n_{k}k")
output_path = Path(f"data/datasets/TON_IoT_{n_nodes}n_{k}k")

df_full = df_encoded.copy()
df_full[target_col] = y_encoded
df_full = df_full.sample(frac=1, random_state=np.random.RandomState()).reset_index(drop=True)
#splits = np.array_split(df_full, n_nodes)
#splits = np.array_split(df_full, 10)

#for i in range(n_simulations):
i = 0
split_df = df_full
dataset_folder = Path(output_path / str(i) / f"4inTestBalancedAlpha0{alphazero}_seedRand")
dataset_folder.mkdir(parents=True, exist_ok=True)
dataset_folder2 = Path(output_path / str(i+1) / f"4inTestBalancedAlpha0{alphazero}_seedRand")
dataset_folder2.mkdir(parents=True, exist_ok=True)

target_col = split_df.columns[-1]
y = split_df[target_col]
x = split_df.drop(columns=[target_col])

node_map, _ = dirichlet_split(y, n_clients=n_nodes, alpha=alpha_value)

for node_id, idxs in node_map.items():
    X_node = x.iloc[idxs]
    Y_node = y.iloc[idxs]

    X_train, Y_train, X_val, Y_val, X_test, Y_test = train_val_test_split(
            X_node, Y_node, test_perc=0.3, val_perc_on_train=0.1
        )

    np.savez(
            dataset_folder / f"node_{node_id}.npz",
            X_train=X_train.to_numpy(),
            Y_train=Y_train.to_numpy(),
            X_val=X_val.to_numpy(),
            Y_val=Y_val.to_numpy(),
            X_test=X_test.to_numpy(),
            Y_test=Y_test.to_numpy(),
        )
    
    np.savez(
            dataset_folder2 / f"node_{node_id}.npz",
            X_train=X_train.to_numpy(),
            Y_train=Y_train.to_numpy(),
            X_val=X_val.to_numpy(),
            Y_val=Y_val.to_numpy(),
            X_test=X_test.to_numpy(),
            Y_test=Y_test.to_numpy(),
        )
    #split_df.to_parquet(dataset_folder / f"node_{node_id}.parquet", index=False)
    print(f"Node {node_id} in simulation {i} saved!")
print(f"Simulation {i} saved!")


In [ ]:
#Reset del Kernel
%reset -s -f

#import os
#os._exit(00)


#from IPython.core.display import HTML
#HTML("<script>Jupyter.notebook.kernel.restart()</script>")

In [ ]:
import functools
import json
import os
from pathlib import Path
import torch
import torch.nn as nn

from gossiplearning.config import Config
from utils.gossip_training import get_node_dataset, round_trip_fn, model_transmission_fn, \
    run_simulation
from utils.model_creators import create_MLP
from utils.multiprocessing_test import run_in_parallel

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
import tensorflow as tf

n_nodes = 10
k=3
n_simulations = 10
alphazero = 2
#n_nodes = 2
#k=1
#n_simulations = 1
network_path = Path(f"data/networks/TON_IoT_{n_nodes}n_{k}k")
output_path = Path(f"data/datasets/TON_IoT_{n_nodes}n_{k}k")

In [ ]:
with open("TON_Iot_config.json", "r") as f:
    config = json.load(f)

config = Config.model_validate(config)

workspace_path = Path(config.workspace_dir)
workspace_path.mkdir(exist_ok=True, parents=True)
with (workspace_path / "config.json").open("w") as f:
    json.dump(config.model_dump(), f, indent=True)

In [ ]:
from utils.data import get_common_test_set
from gossiplearning.weight import weight_by_dataset_size

model_creator = functools.partial(create_MLP, config=config)

run_in_parallel(
    [
        functools.partial(
           run_simulation,
           config=config,
           simulation_number=i,
           network_folder=network_path / str(i),
           round_trip_fn=round_trip_fn,
           model_transmission_fn=model_transmission_fn,
           node_data_fn=functools.partial(
               get_node_dataset, 
               base_folder=output_path, 
               simulation_number=i, 
               ds_name=f"4inTestBalancedAlpha0{alphazero}_seedRand"
           ),
           model_creator=model_creator,
           get_test_set = functools.partial(
                get_common_test_set,
                node_data_fn=functools.partial(
                                get_node_dataset,
                                base_folder=output_path, 
                                simulation_number=i, 
                                ds_name=f"4inTestBalancedAlpha0{alphazero}_seedRand"
                            ),
                n_nodes=config.n_nodes,
                perc=0.1
            ),
           weight_fn=weight_by_dataset_size
        )
        for i in range(2)
    ]
)